In [1]:
# NEWS API
# https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo

import requests as r
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
AV_KEY = os.getenv("AV_KEY")

In [6]:
symbol = "AEP"
# news api
time_from = "20160101T0000"
time_to = "20251231T2359"
sort = "RELEVANCE"
url_base = "https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
limit = 1000
url = f"{url_base}&tickers={symbol}&time_from={time_from}&time_to={time_to}&sort={sort}&limit={limit}&apikey={AV_KEY}"
res = r.get(url)

data = res.json()
df = pd.DataFrame.from_dict(data['feed'])

# order by timepublished
df = df.sort_values(by="time_published", ascending=False)

# change the time_published to datetime
df["time_published"] = pd.to_datetime(df["time_published"], errors="coerce")
df

,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment
0,"Cwm LLC Acquires 234,689 Shares of American El...",https://www.marketbeat.com/instant-alerts/fili...,2025-12-31 17:13:37,[MarketBeat],Cwm LLC significantly increased its stake in A...,https://www.marketbeat.com/logos/american-elec...,MarketBeat,General,MarketBeat,"[{'topic': 'earnings', 'relevance_score': '0.9...",0.132877,Neutral,"[{'ticker': 'AEPPZ', 'relevance_score': '0.327..."
957,"American Electric Power Company, Inc. $AEP Sha...",https://www.marketbeat.com/instant-alerts/fili...,2025-12-31 08:47:55,[MarketBeat],Allspring Global Investments trimmed its stake...,https://www.marketbeat.com/logos/american-elec...,MarketBeat,General,MarketBeat,"[{'topic': 'earnings', 'relevance_score': '0.9...",0.199724,Somewhat-Bullish,"[{'ticker': 'AEP', 'relevance_score': '1.00000..."
1,NiSource Stock Holds Its Nerve as Utilities Re...,https://www.ad-hoc-news.de/boerse/ueberblick/n...,2025-12-30 16:08:07,[],"NiSource Inc. has shown strong performance, ou...",https://www.ad-hoc-news.de/img/logos/logo_264x...,AD HOC NEWS,General,AD HOC NEWS,"[{'topic': 'earnings', 'relevance_score': '0.8...",0.377529,Bullish,"[{'ticker': 'NI', 'relevance_score': '1.000000..."
303,How two nuclear plants quietly power a quarter...,https://www.stocktitan.net/news/SO/georgia-mar...,2025-12-30 14:40:26,[NULL],Georgia Power and Southern Nuclear are celebra...,https://mma.prnewswire.com/media/128122/Georgi...,Stock Titan,General,Stock Titan,"[{'topic': 'energy_transportation', 'relevance...",0.126709,Neutral,"[{'ticker': 'SO', 'relevance_score': '1.000000..."
2,New nuclear option proposed to power growing c...,https://www.stocktitan.net/news/DUK/duke-energ...,2025-12-30 14:00:17,[NULL],Duke Energy has submitted an early site permit...,https://mma.prnewswire.com/media/660545/Duke_E...,Stock Titan,General,Stock Titan,"[{'topic': 'energy_transportation', 'relevance...",0.088682,Neutral,"[{'ticker': 'DUK', 'relevance_score': '1.00000..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
701,"NiSource Names Pablo A. Vegas, Executive Vice ...",https://www.prnewswire.com/news-releases/nisou...,2016-03-15 10:00:00,[],NiSource Inc. announced the appointment of Pab...,http://photos.prnewswire.com/prnh/20160314/344189,PR Newswire,General,PR Newswire,"[{'topic': 'finance', 'relevance_score': '0.70...",0.323252,Somewhat-Bullish,"[{'ticker': 'NI', 'relevance_score': '1.000000..."
763,Edenor es responsable de forma objetiva por lo...,https://aldiaargentina.microjuris.com/2016/03/...,2016-03-10 06:21:41,[],"A court ruling confirms Edenor, a public servi...",NaN,al día | argentina,General,al día | argentina,"[{'topic': 'finance', 'relevance_score': '0.81...",-0.287839,Somewhat-Bearish,"[{'ticker': 'EDN', 'relevance_score': '1.00000..."
702,Oil exec Aubrey McClendon dies in car crash a ...,https://www.cbsnews.com/news/ex-ceo-of-chesape...,2016-03-02 19:51:00,[CBS/AP],"Aubrey McClendon, former chief executive of Ch...",NaN,CBS News,General,CBS News,"[{'topic': 'energy_transportation', 'relevance...",0.084146,Neutral,"[{'ticker': 'EXE', 'relevance_score': '1.00000..."
956,Consent approved for the Hinkley Point C conne...,https://www.gov.uk/government/news/consent-app...,2016-01-19 07:06:24,[],The Department for Energy and Climate Change h...,NaN,GOV.UK,General,GOV.UK,"[{'topic': 'energy_transportation', 'relevance...",0.335462,Somewhat-Bullish,"[{'ticker': 'NGG', 'relevance_score': '1.00000..."


In [5]:
# import libraries and define helper functions for fetching news data from Alpha Vantage API
from datetime import datetime
from pathlib import Path
import time
import pandas as pd
import requests as r

def _format_av_time(dt: datetime) -> str:
    return dt.strftime("%Y%m%dT%H%M")


def _request_news_window(symbol: str, start_dt: datetime, end_dt: datetime, api_key: str, limit: int = 1000, sort: str = "LATEST"):
    url = (
        "https://www.alphavantage.co/query"
        "?function=NEWS_SENTIMENT"
        f"&tickers={symbol}"
        f"&time_from={_format_av_time(start_dt)}"
        f"&time_to={_format_av_time(end_dt)}"
        f"&sort={sort}"
        f"&limit={limit}"
        f"&apikey={api_key}"
    )
    response = r.get(url, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if "Information" in payload or "Note" in payload or "Error Message" in payload:
        raise RuntimeError(payload.get("Information") or payload.get("Note") or payload.get("Error Message"))
    return payload.get("feed", [])


def fetch_all_news_for_ticker(
    symbol: str,
    start: str = "2016-01-01T00:00",
    end: str = "2025-12-31T23:59",
    api_key: str | None = AV_KEY,
    limit: int = 1000,
    sort: str = "RELEVANCE",
    min_window_minutes: int = 60,
    pause_seconds: float = 12.5,
):
    if not api_key:
        raise ValueError("AV_KEY is missing. Set it in your environment before running this cell.")

    start_dt = datetime.fromisoformat(start)
    end_dt = datetime.fromisoformat(end)
    if start_dt >= end_dt:
        raise ValueError("start must be earlier than end")

    stack = [(start_dt, end_dt)]
    items = []

    while stack:
        window_start, window_end = stack.pop()
        feed = _request_news_window(symbol, window_start, window_end, api_key=api_key, limit=limit, sort=sort)

        if not feed:
            continue

        if len(feed) < limit:
            items.extend(feed)
            continue

        window_minutes = max(int((window_end - window_start).total_seconds() // 60), 0)
        if window_minutes <= min_window_minutes:
            items.extend(feed)
            continue

        midpoint = window_start + (window_end - window_start) / 2
        midpoint = midpoint.replace(second=0, microsecond=0)

        if midpoint <= window_start or midpoint >= window_end:
            items.extend(feed)
            continue

        stack.append((midpoint, window_end))
        stack.append((window_start, midpoint))

        if pause_seconds:
            time.sleep(pause_seconds)

    seen = set()
    unique_items = []
    for item in items:
        key = item.get("url") or f"{item.get('title', '')}|{item.get('time_published', '')}"
        if key in seen:
            continue
        seen.add(key)
        unique_items.append(item)

    return pd.DataFrame(unique_items)


tickers = ['AEP']
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

for ticker in tickers:
    df_all_news = fetch_all_news_for_ticker(
        symbol=ticker,
        start="2016-01-01T00:00",
        end="2025-12-31T23:59",
        api_key=AV_KEY,
        sort="RELEVANCE",
    )

    if not df_all_news.empty and "time_published" in df_all_news.columns:
        df_all_news["time_published"] = pd.to_datetime(df_all_news["time_published"], errors="coerce")
        df_all_news = df_all_news.sort_values("time_published").reset_index(drop=True)

    output_path = output_dir / f"{ticker}.parquet"
    df_all_news.to_parquet(output_path, index=False)
    print(f"Saved {len(df_all_news)} rows to {output_path}")

Saved 1000 rows to data\AEP.parquet
